In [0]:
# lecture Silver
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA silver")

df_silver=spark.table("transactions_silver")

In [0]:
# KPI globaux
from pyspark.sql import functions as F

df_kpi = (
    df_silver
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("amount").alias("total_amount"),
        F.sum(F.col("is_fraud")).alias("nb_fraud"),
        F.sum(F.when(F.col("is_fraud") == 1, F.col("amount")).otherwise(0)).alias("fraud_amount"),
        F.sum(F.when(F.col("is_fraud") == 0, F.col("amount")).otherwise(0)).alias("legit_amount"),
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)


In [0]:
# Enrichissement Time en miutes / heures /jour
from pyspark.sql import functions as F

df_gold_base = (
    df_silver
    .withColumn("minute", (F.col("time") / 60).cast("int"))
    .withColumn("hour", (F.col("time") / 3600).cast("int"))
    .withColumn("day", (F.col("time") / 86400).cast("int"))
)


In [0]:
## Fraude par minute
df_fraud_by_minute = (
    df_gold_base
    .groupBy("minute")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

## Fraude par heure
df_fraud_by_hour = (
    df_gold_base
    .groupBy("hour")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

## Fraude par jour
df_fraud_by_day = (
    df_gold_base
    .groupBy("day")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

In [0]:
## score de risque basé sur le montant
df_risk = (
    df_gold_base
    .withColumn(
        "risk_score",
        F.when(F.col("amount") > 2000, 0.9)
         .when(F.col("amount") > 1000, 0.7)
         .when(F.col("amount") > 500, 0.5)
         .otherwise(0.2)
    )
)

In [0]:
# Écriture dans Unity Catalog
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA gold")

df_kpi.write.format("delta").mode("overwrite").saveAsTable("fraud_kpi_gold")
df_fraud_by_minute.write.format("delta").mode("overwrite").saveAsTable("fraud_by_minute_gold")
df_fraud_by_hour.write.format("delta").mode("overwrite").saveAsTable("fraud_by_hour_gold")
df_fraud_by_day.write.format("delta").mode("overwrite").saveAsTable("fraud_by_day_gold")
df_risk.write.format("delta").mode("overwrite").saveAsTable("fraud_risk_gold")